# Certifying fault tolerance

This notebook shows how to compute exact certificates of fault tolerance for noisy Stim circuits.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
import stim
import sympy

from qldpc import circuits, codes

## Certify the fault distance of a memory experiment

The fault distance of a noisy circuit is the smallest number of error mechanisms that together flip a logical observable without firing any detector.
Enumerating every combination of error mechanisms is impractical beyond small circuits: the bivariate bicycle example below has 432 error mechanisms after merging equivalent ones, so checking every combination of up to 6 of them would take $\sum_{k=1}^{6} \binom{432}{k} \approx 9 \times 10^{12}$ steps.
`circuits.get_fault_distance` instead converts the circuit into an auxiliary CSS code and uses the Brouwer-Zimmermann code-distance algorithm, which stops as soon as it can prove that no smaller combination of error mechanisms flips a logical observable without firing any detector.
`circuits.get_fault_distance` thereby computes the fault distance exactly in a few seconds here, and it scales to much larger circuits, though its cost is still exponential in the fault distance.
See `help(circuits.get_fault_distance)` for additional information.

Here we compute the fault distance of X- and Z-basis memory experiments for the [[72,12,6]] bivariate bicycle code from [arXiv:2308.07915](https://arxiv.org/abs/2308.07915), with one round of syndrome extraction under circuit-level depolarizing noise.

In [3]:
x, y = sympy.symbols("x y")
code = codes.BBCode({x: 6, y: 6}, x**3 + y + y**2, y**3 + x + x**2)
noise_model = circuits.DepolarizingNoiseModel(1e-3)


def get_noisy_memory_circuit(code, basis, num_rounds=1):
    # one logical memory cycle = num_rounds rounds of syndrome extraction
    # you can replace logical_memory_cycle with another memory circuit
    initialization, logical_memory_cycle, readout, *_ = circuits.get_memory_experiment_parts(
        code, basis=basis, num_rounds=num_rounds
    )

    # keep initialization and readout noiseless, to certify only the inner QEC cycle
    return initialization + noise_model.noisy_circuit(logical_memory_cycle) + readout


print(f"code parameters: {code.get_code_params()}")
for basis in ("X", "Z"):
    noisy_circuit = get_noisy_memory_circuit(code, basis)
    num_errors = noisy_circuit.detector_error_model().num_errors
    fault_distance = circuits.get_fault_distance(noisy_circuit)
    print(f"{basis}-basis memory: {num_errors} error mechanisms, fault distance {fault_distance}")

code parameters: (72, 12, 6)
X-basis memory: 432 error mechanisms, fault distance 6
Z-basis memory: 432 error mechanisms, fault distance 6


### The Steane code

The above example uses bare-ancilla syndrome extraction.
However, this syndrome measurement strategy does not always preserve the distance of a code.
In the case of the Steane code, the fault distance of the bare-ancilla syndrome measurement circuit is less than the distance of the Steane code, indicating that other (e.g., flagged, Shor, or Steane-type QEC) strategies are necessary for fault-tolerant QEC.

In [4]:
code = codes.SteaneCode()

print(f"code parameters: {code.get_code_params()}")
for basis in ("X", "Z"):
    noisy_circuit = get_noisy_memory_circuit(code, basis)
    print(f"{basis}-basis memory: fault distance {circuits.get_fault_distance(noisy_circuit)}")

code parameters: (7, 1, 3)
X-basis memory: fault distance 2
Z-basis memory: fault distance 2


## Certify fault-tolerant state preparation

`circuits.certify_bounded_fault_propagation` checks that a noisy circuit has bounded fault propagation.
An error mechanism is one error instruction of the circuit's detector error model.
A fault is a set of error mechanisms, and its fault weight is the number of error mechanisms in it.
Fault propagation is bounded if every fault leaves an output error that is equivalent, up to a stabilizer of the prepared state, to an error whose Pauli weight is at most the fault weight.
The Pauli weight of an error is either its physical weight (the number of qubits that it acts on) or its CSS weight $\max(|x|, |z|)$, where $x$ and $z$ are its X and Z support vectors.
By default, the certifier uses CSS weight for a CSS state and physical weight otherwise, and `css_weight` overrides this choice.
Bounded fault propagation is a sufficient criterion for fault-tolerant state preparation; see Definition 3 in Section 4.2 of [arXiv:0904.2557](https://arxiv.org/abs/0904.2557).
See `help(circuits.certify_bounded_fault_propagation)` for additional information.

Here we prepare GHZ states by fanning out CX gates, and add circuit-level depolarizing noise with a noise model.

In [5]:
noise_model = circuits.DepolarizingNoiseModel(1e-3)

for num_qubits in (3, 4):
    ghz_circuit = stim.Circuit("H 0")
    for target in range(1, num_qubits):
        ghz_circuit.append("CX", [0, target])
    noisy_circuit = noise_model.noisy_circuit(ghz_circuit)

    certificate = circuits.certify_bounded_fault_propagation(noisy_circuit)
    print(f"{num_qubits}-qubit GHZ state: certified = {certificate.certified}")
    if counterexample := certificate.counterexample:
        pauli = counterexample.output_pauli
        bound = counterexample.output_weight_lower_bound
        indices = counterexample.fault_indices
        print(f"  fault {indices}: output error equivalent to {pauli}")
        print(f"  equivalent output errors have Pauli weight at least {bound}")

3-qubit GHZ state: certified = True
4-qubit GHZ state: certified = False
  fault (4,): output error equivalent to +ZXX_
  equivalent output errors have Pauli weight at least 2


With four qubits, a fault of a single error mechanism already leaves an error that no stabilizer reduces to Pauli weight at most one, so the certificate fails and returns that fault as a counterexample.
`fault_indices` lists the error mechanisms of the fault by their index among the error instructions of `certificate.detector_error_model`.

### Post-selection

Repeat-until-success protocols discard attempts that fail post-selection, so only faults that pass post-selection, meaning that they fire none of the selected detectors, must satisfy that bound.
Here we flag the preparation of the Steane code's logical $|0\rangle$ state.
`add_flags=True` adds a detector for each measurement that no detector addresses, and `post_select="flags"` post-selects on those flags.
Post-selection is required for the fault-tolerance of this protocol.

In [6]:
prep_circuit = stim.Circuit("""
    # non-fault-tolerant preparation of the logical zero state
    H 0 2 4
    CX 0 3 2 1 4 5
    CX 0 1 2 6 4 3
    CX 2 5 3 6
    # flag qubit 7 detects errors in a logical Z representative
    H 7
    CZ 7 1 7 3 7 5
    MX 7
""")
noisy_circuit = circuits.DepolarizingNoiseModel(1e-3).noisy_circuit(prep_circuit)

for post_select in (False, "flags"):
    certificate = circuits.certify_bounded_fault_propagation(
        noisy_circuit, add_flags=True, post_select=post_select
    )
    print(f"post_select={post_select!r}: certified = {certificate.certified}")
    if counterexample := certificate.counterexample:
        pauli = counterexample.output_pauli
        bound = counterexample.output_weight_lower_bound
        indices = counterexample.fault_indices
        print(f"  fault {indices}: output error equivalent to {pauli}")
        print(f"  equivalent output errors have Pauli weight at least {bound}")

post_select=False: certified = False
  fault (1,): output error equivalent to +YY_Z___
  equivalent output errors have Pauli weight at least 2
post_select='flags': certified = True
